# Notebook 02 — Classification Analysis

Behavioural feature exploration + genotype classification. Pool multiple tracking runs (same day / genotype layout) for more flies per genotype.

## Stages
1. Configuration (`RUN_DIRS`, `CV_MODE`, auto-incremented `FIGURES_DIR`)
2. Load `ordered_tracks.csv` per run, map genotypes, concat with run-scoped `ordered_id`
3. Extract frame-level behavioural features
4. Aggregate to per-fly features (+ `run` column)
5. Feature list + **per-run sanity** box plots (catch video/batch offsets)
6. Exploratory plots (by genotype, WT vs mutant)
7. Classification — **one** algorithm from `config.yaml` (`classification.method`: svc | lda | logistic): **stratified** CV or **group** CV (leave-one-video-out)

**Edit the configuration cell** (`RUN_DIRS`, `CV_MODE`).

In [ ]:
import sys
sys.path.insert(0, '..')  # so 'src' is importable from notebooks/

import os
import pandas as pd
import plotly.express as px

from src.classification import (
    map_vial_to_genotype,
    run_classifier,
    plot_by_genotype,
    plot_wt_vs_mutant,
    pretty_run_label,
    write_classification_report_site,
    apply_boxplot_trim_for_feature,
    boxplot_trim_title_suffix,
    pooled_genotype_color_map,
)
from src.features import (
    extract_behavioral_features,
    aggregate_per_fly_features,
    aggregate_feature_plot_titles,
    augment_trajectories_geometric,
    augment_trajectories_masking,
    classification_feature_columns,
)
from src.latent_space import run_latent_space_analysis
from utils import load_config

## 1 — Configuration

In [ ]:
# ---- EDIT THIS ----
RUN_DIRS = [
    "../outputs/run_122_6DPE_n001/", 
    "../outputs/run_123_6DPE_n002/", 
    "../outputs/run_124_6DPE_n003/", 
    "../outputs/run_125_6DPE_n004/", 
    "../outputs/run_126_6DPE_n005/", 
    "../outputs/run_127_6DPE_n006/"
]
CV_MODE = "group"  # "stratified" = pool flies | "group" = leave-one-video-out (GroupKFold)
RUN_LATENT_SPACE = True
WRITE_REPORT_SITE = True
assert CV_MODE in ("stratified", "group")

_names = [os.path.basename(d.rstrip("/\\")) for d in RUN_DIRS]
_pretty = [pretty_run_label(n) for n in _names]
tag = _pretty[0] if len(_pretty) == 1 else f"{_pretty[0]}_to_{_pretty[-1]}"

if CV_MODE == "group" and len(RUN_DIRS) < 2:
    print(f"CV_MODE='group' needs >=2 runs; got {len(RUN_DIRS)}. Falling back to 'stratified'.")
    CV_MODE = "stratified"

classif_root = "../outputs/classification"
os.makedirs(classif_root, exist_ok=True)
prefix = f"{tag}_run"
existing = [d for d in os.listdir(classif_root) if d.startswith(prefix)]
nums = [int(d.removeprefix(prefix)) for d in existing if d.removeprefix(prefix).isdigit()]
FIGURES_DIR = os.path.join(classif_root, f"{tag}_run{max(nums, default=0) + 1}")
os.makedirs(FIGURES_DIR, exist_ok=True)
print("Figures dir:", FIGURES_DIR)

# Units + boxplot trim read config at import (src.features / src.classification).
# After editing config.yaml: Kernel -> Restart Session, then Run All.
import src.features as _feat_cfg
print("Behaviour units from config:", dict(_feat_cfg.UNITS))

## 2 — Load data and map genotypes

`map_vial_to_genotype` parses the filename to infer which vial corresponds
to which genotype (e.g. `..._hTDP43_WT-Het-Homo_...`).

In [3]:
parts = []
for rd in RUN_DIRS:
    d = map_vial_to_genotype(rd)
    run_tag = os.path.basename(os.path.normpath(rd))
    d["run"] = run_tag
    d["run_dir"] = rd
    d["ordered_id"] = run_tag + "::" + d["ordered_id"].astype(str)
    parts.append(d)
df_raw = pd.concat(parts, ignore_index=True)
print(df_raw.shape, "| runs:", df_raw["run"].nunique())
print(df_raw["genotype"].value_counts())
df_raw.head()

(130363, 10) | runs: 6
genotype
G287S    25806
M337V    22525
G294A    22102
A90V     21815
A315T    21340
WT       16775
Name: count, dtype: int64


,frame,orig_id,x,y,vial_id,ordered_id,fps,genotype,run,run_dir
0,0,id1,291.5,364.5,vial3,run_122_6DPE_n001::40,30.0,G287S,run_122_6DPE_n001,../outputs/run_122_6DPE_n001/
1,1,id1,291.5,364.5,vial3,run_122_6DPE_n001::40,30.0,G287S,run_122_6DPE_n001,../outputs/run_122_6DPE_n001/
2,2,id1,291.5,365.0,vial3,run_122_6DPE_n001::40,30.0,G287S,run_122_6DPE_n001,../outputs/run_122_6DPE_n001/
3,3,id1,291.5,365.0,vial3,run_122_6DPE_n001::40,30.0,G287S,run_122_6DPE_n001,../outputs/run_122_6DPE_n001/
4,4,id1,291.5,365.0,vial3,run_122_6DPE_n001::40,30.0,G287S,run_122_6DPE_n001,../outputs/run_122_6DPE_n001/


## 2b — Geometric augmentation (rows, not columns)

If `classification.augmentation.enabled` is true in `config.yaml`, each listed
transform produces a new copy of every fly's raw (x, y) trajectory with the
same genotype label. The augmented copies feed only the classifier path; the
pristine `df_raw` is preserved for latent-space analysis and box plots.

Each augmented row carries `aug_group = original ordered_id`. When passed to
`run_classifier` as `groups=`, GroupKFold keeps all copies of one fly inside
a single CV fold so no augmented twin leaks across train/test folds.

In [ ]:
_cls_cfg = load_config("../config.yaml").classification
_aug_cfg = getattr(_cls_cfg, "augmentation", None)
AUG_ENABLED = bool(getattr(_aug_cfg, "enabled", False))
AUG_TRANSFORMS = list(getattr(_aug_cfg, "transforms", []) or [])
_mask_cfg = getattr(_cls_cfg, "masking", None)
MASK_ENABLED = bool(getattr(_mask_cfg, "enabled", False))

if AUG_ENABLED and AUG_TRANSFORMS:
    print(f"Augmenting trajectories: {AUG_TRANSFORMS} ({len(AUG_TRANSFORMS)}x rows)")
    df_raw_for_feat = augment_trajectories_geometric(df_raw, transforms=AUG_TRANSFORMS)
else:
    print("Geometric augmentation disabled.")
    df_raw_for_feat = df_raw

if MASK_ENABLED:
    _mf = float(getattr(_mask_cfg, "mask_fraction", 0.10))
    _n = int(round(1.0 / _mf))
    print(f"Applying masking: mask_fraction={_mf} -> {_n} copies per fly")
    _df_masked = augment_trajectories_masking(
        df_raw_for_feat if not AUG_ENABLED else df_raw,
        mask_fraction=_mf,
    )
    df_raw_for_feat = pd.concat([df_raw_for_feat, _df_masked], ignore_index=True)
else:
    print("Masking disabled.")

print("df_raw_for_feat:", df_raw_for_feat.shape)

## 3 — Extract behavioural features

Computes frame-level kinematics (velocity, acceleration, turning angle),
convex-hull area, and path tortuosity for each fly.

In [ ]:
df_feat = extract_behavioral_features(df_raw_for_feat)
print(df_feat.shape)
df_feat[["ordered_id", "frame", "velocity", "turning_angle", "area_covered", "tortuosity"]].head()

## 4 — Aggregate to per-fly features

In [ ]:
df_agg = aggregate_per_fly_features(df_feat, pause_threshold=1.0)

meta = (
    df_raw_for_feat.drop_duplicates("ordered_id")
    .set_index("ordered_id")[["genotype", "run"]]
)
df_agg = df_agg.join(meta, on="ordered_id").dropna(subset=["genotype"])

# Carry aug_group (original ordered_id) through aggregation so GroupKFold can
# keep all augmented copies of one fly in a single CV fold.
if "aug_group" in df_raw_for_feat.columns:
    _ag = df_raw_for_feat.drop_duplicates("ordered_id").set_index("ordered_id")["aug_group"]
    df_agg["aug_group"] = df_agg["ordered_id"].map(_ag)

# Pristine subset: original rows only. Use for boxplots and the report site so
# duplicated trajectories don't inflate per-genotype distributions.
df_agg_orig = (
    df_agg[df_agg["ordered_id"] == df_agg["aug_group"]].copy()
    if "aug_group" in df_agg.columns
    else df_agg
)

print(df_agg.shape, "| original rows:", df_agg_orig.shape[0])
df_agg.head()

In [ ]:
# Full list from config (``features.kinematic_three_families`` adds x/y/mag families).
FEATURES = classification_feature_columns()

# Titles follow calibration.* in config.yaml (cm/s, px/s, etc.) - not hardcoded here.
FEATURE_TITLES = aggregate_feature_plot_titles(FEATURES)

hover_data = ["ordered_id", "run"]

# One genotype → color map for every Plotly figure (matches plot_by_genotype / latent 3D).
# Built from the original-only subset so augmented twins don't perturb category order.
GENOTYPE_COLOR_MAP = pooled_genotype_color_map(df_agg_orig)
GENOTYPE_CATEGORY_ORDER = list(GENOTYPE_COLOR_MAP.keys())

In [ ]:
# Per-run sanity check: batch / lighting effects should not dwarf genotype.
# Uses df_agg_orig so augmented copies don't appear in the per-run distribution.
# x-axis shows the prettified run label (e.g. 6DPE_001) not the verbose dir name.
if len(RUN_DIRS) > 1:
    _dplot_base = df_agg_orig.copy()
    _dplot_base["run_label"] = _dplot_base["run"].map(pretty_run_label)
    _run_order = sorted(_dplot_base["run_label"].unique())
    for feat in FEATURES:
        dplot = apply_boxplot_trim_for_feature(_dplot_base, feat)
        fig = px.box(
            dplot,
            x="run_label",
            y=feat,
            color="genotype",
            points="all",
            hover_data=hover_data,
            category_orders={"genotype": GENOTYPE_CATEGORY_ORDER, "run_label": _run_order},
            color_discrete_map=GENOTYPE_COLOR_MAP,
            title=FEATURE_TITLES[feat],
        )
        fig.update_traces(jitter=0.3, marker=dict(size=6, opacity=0.75))
        fig.update_layout(yaxis_title=FEATURE_TITLES[feat], xaxis_title="Run")
        fig.write_html(os.path.join(FIGURES_DIR, f"{feat}_per_run.html"))
        fig.show()

## 5 — Exploratory visualisation

Box plots for each feature, grouped by genotype (uses `FEATURES` / `hover_data` from above).

In [ ]:
plot_by_genotype(df_agg_orig, FEATURES, FEATURE_TITLES, hover_data, outdir=FIGURES_DIR)

In [ ]:
plot_wt_vs_mutant(df_agg_orig, FEATURES, FEATURE_TITLES, hover_data, outdir=FIGURES_DIR)

## 6 - Classification

Exactly **one** classifier is used end-to-end (notebook, CLI, saved figures, and the HTML report): whichever you set in `classification.method` (`svc` | `lda` | `logistic`). The matching hyperparameter block in `classification:` is used; the other two blocks are ignored until you switch `method`. Defaults to SVC with an RBF kernel.

`CV_MODE == "group"` uses **GroupKFold** (no fly from a held-out video in training). Needs **>=2 runs**.  
`CV_MODE == "stratified"` pools all flies across videos.

For SVC, set `classification.svc.grid_search.enabled: true` to grid-search over the `C` and `gamma` lists in config. The best combination is printed and stamped onto the CV figure title.

Figures go to `FIGURES_DIR` (auto-incremented under `outputs/classification/`).

In [ ]:
# Pick groups for CV.
# Priority: if augmentation is on, use aug_group so augmented twins of one
# fly stay in a single fold (no leakage across train/test). Otherwise honor
# CV_MODE: "group" = leave-one-video-out via run, "stratified" = pool flies.
if "aug_group" in df_agg.columns:
    groups = df_agg["aug_group"].to_numpy()
    _scheme_msg = (
        f"augmentation on -> GroupKFold by aug_group "
        f"({df_agg['aug_group'].nunique()} unique flies, "
        f"{len(df_agg)} rows incl. augmented copies)"
    )
elif CV_MODE == "group":
    groups = df_agg["run"].to_numpy()
    _scheme_msg = f"group ({df_agg['run'].nunique()} video groups, GroupKFold)"
else:
    groups = None
    _scheme_msg = "stratified"
print(f"CV scheme: {_scheme_msg}")

# Active method (svc | lda | logistic) and its hyperparameters live in
# config.yaml under "classification:". For SVC, set
# classification.svc.grid_search.enabled: true to grid-search C and gamma;
# the best combo is printed below and stamped onto the CV figure title.
for mode in ["multiclass", "binary"]:
    print(f"\n=== classification [{mode}] ===")
    run_classifier(
        df=df_agg,
        outdir=FIGURES_DIR,
        classification_mode=mode,
        cv=5,
        plot_importance=True,
        groups=groups,
    )

## 7 — Report site (`classification_report.html`) + optional latent-space page

Build a navigable report with a sidebar entry page and modular section files (pooled + per-trial pages) to avoid huge single-file scrolling. If enabled, latent-space outputs are exported and linked into the same navigation hub.

In [ ]:
latent_rel = None
if RUN_LATENT_SPACE:
    latent = run_latent_space_analysis(df_raw)
    latent_dir = os.path.join(FIGURES_DIR, "latent_space")
    os.makedirs(latent_dir, exist_ok=True)

    _method = latent["analysis1"].get("embedding", {}).get("method", "embedding")
    latent["analysis1"]["embedding_fig"].write_html(os.path.join(latent_dir, f"{_method}_xy_kinematics.html"))
    latent["analysis2"]["embedding_fig"].write_html(os.path.join(latent_dir, f"{_method}_hist_kinematics.html"))
    latent["rf_importance_fig"].write_html(os.path.join(latent_dir, "rf_importance.html"))

    p1a = latent["analysis1"]["permanova"]["run_aware"]
    p1b = latent["analysis1"]["permanova"]["pooled"]
    p2a = latent["analysis2"]["permanova"]["run_aware"]
    p2b = latent["analysis2"]["permanova"]["pooled"]

    latent_page = os.path.join(FIGURES_DIR, "latent_space_report.html")
    with open(latent_page, "w", encoding="utf-8") as f:
        f.write("\n".join([
            "<!DOCTYPE html><html lang='en'><head><meta charset='utf-8'/><title>Latent-space report</title>",
            "<style>body{font-family:system-ui,sans-serif;margin:1rem 2rem;max-width:1200px;} .card{background:#f7f7f8;padding:0.6rem 0.8rem;border-radius:6px;margin:0.6rem 0;} iframe{width:100%;height:560px;border:1px solid #ddd;margin:0.5rem 0 1.2rem;}</style>",
            "</head><body>",
            "<h1>Latent-space report</h1>",
            f"<div class='card'><strong>Analysis 1 PERMANOVA (run-aware, primary)</strong>: method={p1a['method']}, pseudo-F={p1a['pseudo_f']:.4f}, p={p1a['p_value']:.4g}, R2={p1a['r2']:.4f}</div>",
            f"<div class='card'><strong>Analysis 1 PERMANOVA (pooled, exploratory)</strong>: method={p1b['method']}, pseudo-F={p1b['pseudo_f']:.4f}, p={p1b['p_value']:.4g}, R2={p1b['r2']:.4f}</div>",
            f"<div class='card'><strong>Analysis 2 PERMANOVA (run-aware, primary)</strong>: method={p2a['method']}, pseudo-F={p2a['pseudo_f']:.4f}, p={p2a['p_value']:.4g}, R2={p2a['r2']:.4f}</div>",
            f"<div class='card'><strong>Analysis 2 PERMANOVA (pooled, exploratory)</strong>: method={p2b['method']}, pseudo-F={p2b['pseudo_f']:.4f}, p={p2b['p_value']:.4g}, R2={p2b['r2']:.4f}</div>",
            f"<h2>Fly trajectory and kinematic embedding (3D, {_method})</h2><iframe src='latent_space/{_method}_xy_kinematics.html'></iframe>",
            f"<h2>Fly kinematic distribution embedding (3D, {_method})</h2><iframe src='latent_space/{_method}_hist_kinematics.html'></iframe>",
            "<h2>Random Forest importance of kinematic histogram bins</h2><iframe src='latent_space/rf_importance.html'></iframe>",
            "</body></html>",
        ]))
    latent_rel = "latent_space_report.html"

if WRITE_REPORT_SITE:
    # Report site uses original-only rows: its embedded classifiers and box
    # plots describe biology, not augmented data multiplicity.
    _groups_report = df_agg_orig["run"].values if CV_MODE == "group" else None
    report_entry = write_classification_report_site(
        df=df_agg_orig,
        features=FEATURES,
        feature_titles=FEATURE_TITLES,
        hover_data=hover_data,
        out_dir=FIGURES_DIR,
        trial_column="run",
        report_title=f"{tag}, classification and latent space",
        pooled_cv=5,
        pooled_cv_groups=_groups_report,
        per_trial_cv=5,
        entry_filename="classification_report.html",
        latent_page_filename=latent_rel,
    )
    print("Wrote", os.path.abspath(report_entry))
else:
    print("WRITE_REPORT_SITE=False. Skipped navigable report export.")

## Summary

Per-figure exports (HTML + PNG) are under `FIGURES_DIR`. The report entry page is `classification_report.html`, which links pooled/per-trial sections and (if enabled) latent-space outputs without forcing one giant single-page HTML.